# 第18課：用加密收據保障AI代理安全

## 實作筆記本

這份筆記本包含四個練習：

1. <strong>簽署你的第一張收據</strong>以記錄代理工具呼叫並驗證它。
2. <strong>篡改收據</strong>並觀察驗證失敗。
3. <strong>建立三收據鏈</strong>並確認鏈條完整性。
4. **包裝 Microsoft Agent Framework 工具呼叫**，讓每個操作都生成收據。

所有加密原語均來自維護良好的程式庫（Ed25519 使用 `pynacl`，RFC 8785 標準 JSON 使用 `jcs`，Python 標準庫中的 `hashlib` 用於 SHA-256）。收據邏輯本身是純 Python，可以閱讀與修改。

請依序執行每個程式碼區塊。每個部分都短且自成一格。


## 設定

安裝這兩個依賴。兩者皆採用寬鬆授權（Apache-2.0 / MIT）。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## 輔助工具

這兩個輔助工具負責 base64url 編碼（無填充）和任意物件的 SHA-256 雜湊。它們讓筆記本的其他部分專注於收據邏輯本身。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 第一節：簽署你的第一張收據

想像一下，我們為<strong>Contoso Travel</strong>的代理剛剛為客戶查詢了從悉尼到洛杉磯的航班。我們想將這次工具調用記錄為已簽署的收據，以便未來的審計員可以在不信任我們的情況下驗證它。

### 步驟 1.1：產生簽署金鑰

在生產環境中，代理的簽署金鑰會存放在硬體安全模組（HSM）、Azure Key Vault 或類似的受保護存儲中。此次課程我們在記憶體中產生一把新金鑰。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 第 1.2 步：建立收據有效負載

有效負載包含我們希望收據證明的所有內容：誰操作、使用了什麼工具、帶了哪些參數、返回了什麼結果、遵從了哪項政策，以及發生的時間。我們對參數和結果做雜湊處理，而非直接包含，以免收據洩露敏感內容。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 第 1.3 步：簽署並組合收據

三個步驟：

1. 使用 JCS 標準化有效負載，使兩個實現產生相同邏輯的收據時，生成位元組完全相同的資料。
2. 使用 SHA-256 對標準化後的位元組進行雜湊。
3. 使用 Ed25519 私鑰對雜湊進行簽署。

然後將簽名附加到原始有效負載，產生最終收據。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 第 1.4 步：驗證收據

驗證是逆向的過程。我們剝離簽名，重新計算標準雜湊，並根據收據中的公鑰檢查簽名。

進行此驗證的審計員只需收據本身。不需呼叫任何服務，不需查詢任何鑰匙目錄，也不需任何信任。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

你應該會見到 `Receipt is valid: True`。代理程式已生成其第一個加密簽名的審計記錄。


## 第2節：篡改收據

收據的全部重點在於它們能顯示篡改痕跡。讓我們來證明這一點。

我們將修改收據中的一個字元，並觀察驗證失敗。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 剛才發生了甚麼事？

當我們更改了 `policy_id`，標準字節就改變了。那些字節的 SHA-256 雜湊值隨之改變。簽名（針對原始雜湊產生）便不再符合新的雜湊。驗證正確返回 `False`。

除非攻擊者擁有私鑰，否則無法修改收據的任何欄位並仍通過驗證。只要私鑰存放於密鑰庫中並且公開了公鑰，偽造無法隱藏。

自己試試看：修改上面儲存格中的 `tool_name`、`agent_id` 或 `timestamp` 並重新執行。每次修改都會產生無效的收據。


## 第三節：將收據串連起來

單一收據保護一個操作。大多數代理人會執行多個操作。為了使整個序列具備篡改跡象，我們透過在新收據的有效載荷中包含前一收據的雜湊值，將每個收據連接到前一個收據。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

如果有人移除或重新排序收據，鏈條會在那一點斷裂。任何後續收據的驗證都會失敗，因為其 `previous_receipt_hash` 不再與其前一項的實際雜湊值相符。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

現在通過篡改中間收據並重新驗證來破壞鏈。被篡改的收據未通過其簽名檢查，且下一張收據未通過其鏈接檢查（因為其 `previous_receipt_hash` 不再匹配被修改的中間收據的雜湊值）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

收據 0 仍然通過驗證（它未被修改且沒有前序依賴）。收據 1 未通過簽名檢查，因為我們更改了 `tool_args_hash`。收據 2 未通過鏈接檢查，因為它的 `previous_receipt_hash` 是基於原始（現已修改）收據 1 計算的。

即使攻擊者重新簽署了修改過的收據 1（他們在沒有私鑰的情況下無法做到），收據 2 的鏈接不匹配仍會揭露篡改。為了隱藏更改，攻擊者必須從修改點開始重新簽署每一張收據，這需要擁有私鑰。


## 第4節：用收據簽署包裹代理工具調用

在真實部署中，您不希望每個代理作者都記得調用 `make_receipt`。您希望每次工具調用時自動進行收據簽署。

這裡是最簡單的模式：一個包裝器類，它接受任何可調用的工具函數並返回一個發出收據的版本。這適應於任何代理框架，包括 Microsoft Agent Framework (`agent_framework.foundry`)。

如果您尚未設置 Microsoft Foundry 項目，下方的本地模擬仍然展示此模式。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### 與 Microsoft Agent Framework 整合

上述的 `ReceiptedTool` 包裝器與框架無關。要在使用 Microsoft Agent Framework 建立的代理中使用它，將被包裝的函數註冊為工具。範例（你會將此模擬替換成真正的 Microsoft Foundry 工具註冊）：

```python
# 顯示整合形態的偽代碼。
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="你是 Contoso 旅行社代理人 …",
#     tools=[receipted_lookup],   # 包裝過嘅工具，唔係原始函數
# )
# response = agent.run("搵六月喺悉尼飛洛杉磯嘅航班。")
#
# # 執行後，代理人所使用嘅每一個工具調用都有簽署嘅收據：
# audit_chain = receipted_lookup.receipts
```

代理框架不需要知道任何關於收據的事情。收據簽署是包裝在工具周圍的，而不是綁定到框架中的。這就是你如何在不重寫代理的情況下，為現有代理代碼添加來源追蹤的方法。


## 回顧與進階挑戰

你已經：

- 生成了一對 Ed25519 密鑰。
- 建立並簽署了一個代理工具調用的收據。
- 僅使用公鑰離線驗證了該收據。
- 篡改了一個收據並觀察到驗證失敗。
- 建立了三個收據的哈希鏈序列。
- 篡改了鏈中間的部分，並觀察到簽名失敗和鏈接失敗。
- 將工具函數包裹並自動簽署收據。

**進階挑戰。** 擴展收據架構，加入 `request_id` 欄位（一個用於分佈式追蹤的 UUID）。更新 `make_receipt` 以包含此欄位，並確認收據仍可端到端驗證。然後修改簽署後的欄位，並確認驗證失敗。這將迫使你深入理解典範編碼的每個位元對簽名的貢獻。

**重要界限。** 收據證明三件事且僅證明三件事：歸因（此鑰匙簽署了此內容）、完整性（自簽署以來內容未被更改）及排序（此收據發生於該收據之後）。它們不證明代理的行為是否正確，不證明 `policy_id` 中命名的政策是否真正被評估，也不證明代理遵守每條規則。收據是基礎。治理是你在基礎上構建的系統。

請再次帶著該界限閱讀課程說明。團隊在使用收據時最常犯的錯誤是假設「我們有收據」就代表「我們已經被治理」。事實並非如此。收據使代理行為可稽核，但不代表其正確。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
